# Association Rule Mining with Apriori
### Unsupervised Learning | Beginner-friendly Jupyter Notebook

**Goal:** Discover products that frequently occur together in supermarket transactions.

We will:
1. Download an online grocery transaction dataset.
2. Convert transactions into a one-hot encoded table.
3. Find frequent itemsets with the Apriori algorithm.
4. Generate and interpret association rules using support, confidence, and lift.
5. Visualize the results.

**Important:** Association rules reveal co-occurrence patterns. They do not demonstrate that purchasing one product causes the purchase of another.

## 1. Install the required libraries

In [1]:
%pip -q install pandas matplotlib mlxtend requests

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.59.1 requires numpy<1.27,>=1.22, but you have numpy 2.5.5 which is incompatible.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.5.5 which is incompatible.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but you have numpy 2.5.5 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.1.1 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.5.5 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import libraries

In [2]:
import csv
import io
import requests
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 100)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\user\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\user\anaconda3\Lib\site-packages

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\user\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\user\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## 3. Download the online dataset

The dataset contains supermarket transactions. Each row represents one shopping basket, and the products in that row were purchased together.

The code checks the download response before parsing the file.

In [ ]:
url = "https://raw.githubusercontent.com/SavioSal/datasets/master/store_data_association_rules.csv"

response = requests.get(url, timeout=30)
response.raise_for_status()

transactions = [
    [item.strip() for item in row if item.strip()]
    for row in csv.reader(io.StringIO(response.text))
]

# Remove empty transactions
transactions = [row for row in transactions if row]

print("Number of transactions:", len(transactions))
print("First five transactions:")
for transaction in transactions[:5]:
    print(transaction)

## 4. Convert transactions into a one-hot encoded table

Apriori expects a table in which each column represents a product. `True` indicates that the product appears in that transaction; `False` indicates that it does not.

In [ ]:
encoder = TransactionEncoder()
encoded_array = encoder.fit(transactions).transform(transactions)

basket = pd.DataFrame(encoded_array, columns=encoder.columns_)

print("Dataset shape:", basket.shape)
display(basket.head())

## 5. Find frequent itemsets with Apriori

`min_support=0.01` means that an itemset must appear in at least 1% of all transactions to be retained. Try changing this threshold and observe how the results change.

In [ ]:
frequent_itemsets = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    "support", ascending=False
).reset_index(drop=True)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(frequent_itemsets.head(15))

## 6. Generate association rules

We use a minimum confidence of 0.20. A rule such as `{bread} → {milk}` means that the algorithm found a pattern in which milk often appears in transactions that contain bread.

In [ ]:
if frequent_itemsets.empty:
    raise ValueError("No frequent itemsets found. Try lowering min_support.")

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.20
)

if rules.empty:
    print("No rules met the confidence threshold. Try lowering min_threshold.")
else:
    rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

    display(
        rules[[
            "antecedents", "consequents",
            "support", "confidence", "lift"
        ]].head(15)
    )

## 7. Interpret the three main metrics

- **Support:** The proportion of all transactions that contain both sides of the rule.
- **Confidence:** Among transactions containing the antecedent, the proportion that also contain the consequent.
- **Lift:** The confidence of the rule divided by the overall support of the consequent.

**Lift interpretation**
- `Lift > 1`: The consequent occurs more often with the antecedent than expected under independence.
- `Lift = 1`: The two sides are independent in this dataset.
- `Lift < 1`: The consequent occurs less often with the antecedent than expected under independence.

## 8. Visualize the association rules

In [ ]:
if not rules.empty:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        rules["support"],
        rules["confidence"],
        s=rules["lift"].clip(lower=0.1) * 30,
        alpha=0.6
    )
    plt.xlabel("Support")
    plt.ylabel("Confidence")
    plt.title("Association Rules: Support vs. Confidence")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("There are no rules to visualize.")

## 9. Display the strongest rules in plain language

The following code prints the ten rules with the highest lift. A high lift can indicate an interesting association, but check support as well: a rule based on very few transactions may be unreliable.

In [ ]:
if not rules.empty:
    for _, rule in rules.head(10).iterrows():
        left = ", ".join(sorted(rule["antecedents"]))
        right = ", ".join(sorted(rule["consequents"]))

        print(f"IF a transaction contains [{left}]")
        print(f"THEN it is also associated with [{right}]")
        print(f"Support:    {rule['support']:.3f}")
        print(f"Confidence: {rule['confidence']:.3f}")
        print(f"Lift:       {rule['lift']:.3f}")
        print("-" * 55)

## 10. Student exercise

Run the notebook three times with the following parameter settings. Change the values in the Apriori and association-rules cells.

| Experiment | Minimum support | Minimum confidence |
|---|---:|---:|
| 1 | 0.01 | 0.20 |
| 2 | 0.02 | 0.30 |
| 3 | 0.05 | 0.50 |

**Questions**
1. What happens to the number of frequent itemsets when minimum support increases?
2. What happens to the number of association rules when minimum confidence increases?
3. Find one rule with lift greater than 1. What does it mean?
4. Can an association rule prove that buying one product causes a customer to buy another? Explain.

### Key takeaway
Association rule mining discovers frequent co-occurrence patterns in unlabeled data. It is useful for market-basket analysis, recommendation systems, and exploratory pattern discovery.